In [ ]:
# imports and notebook setup
from pathlib import Path
import importlib
import json
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp")

import numpy as np
import torch

REPO_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "calvin_experiments" / "calvin_rollout_utils.py").exists()
)
for path in [REPO_ROOT, REPO_ROOT / "robomimic", REPO_ROOT / "calvin" / "calvin_env", REPO_ROOT / "calvin_experiments"]:
    path = str(path)
    if path not in sys.path:
        sys.path.insert(0, path)

import robomimic.envs
import robomimic.utils.file_utils as FileUtils
import robomimic.utils.torch_utils as TorchUtils

import calvin_experiments.calvin_rollout_utils as CalvinRolloutUtils

CalvinRolloutUtils = importlib.reload(CalvinRolloutUtils)
from calvin_experiments.calvin_rollout_utils import (
    articulated_binaries_from_start_state,
    capture_scene_snapshot,
    check_state_difference,
    classify_behavior,
    fixed_scene_robot_from_config,
    is_env_connected,
    load_json_config,
    policy_epoch_from_checkpoint,
    plot_rollouts_from_trace_summaries,
    render_visual_camera,
    run_folder_name,
    reset_env_to_scene_robot,
    resolve_checkpoint_path,
    save_rollout_artifacts,
    seed_everything,
)

CONFIG_DIR = REPO_ROOT / "calvin_experiments" / "configs"
VISUALIZATION_CONFIG_PATH = CONFIG_DIR / "visualization_freiburg_style.json"
CHECKPOINT_PATH = REPO_ROOT / "outputs/calvin/base_policy/calvin_D_base_dp/20260501015147/models/model_epoch_280.pth"
OUTPUT_ROOT = REPO_ROOT / "outputs" / "calvin" / "base_tests"
VIDEO_FPS = 30
SCENE_CONFIG_PATH = CONFIG_DIR / "blocks_hidden.json"
SCENE_CONFIG_NAME = load_json_config(SCENE_CONFIG_PATH)["name"]



In [ ]:
# load policy and environment
checkpoint_path = resolve_checkpoint_path(CHECKPOINT_PATH, REPO_ROOT)
POLICY_EPOCH = policy_epoch_from_checkpoint(checkpoint_path)

device = TorchUtils.get_torch_device(try_to_use_cuda=True)
policy, ckpt_dict = FileUtils.policy_from_checkpoint(
    ckpt_path=str(checkpoint_path),
    device=device,
    verbose=True,
)

env, _ = FileUtils.env_from_checkpoint(
    ckpt_dict=ckpt_dict,
    render=False,
    render_offscreen=True,
    verbose=True,
)
if not is_env_connected(env):
    raise RuntimeError("Loaded env is disconnected immediately after env_from_checkpoint.")
base_env_state = {key: np.asarray(value, dtype=np.float32).copy() for key, value in env.get_state().items()}
loaded_checkpoint_path = checkpoint_path
print(f"Loaded policy and env once from: {loaded_checkpoint_path}")
print(f"Policy epoch: {POLICY_EPOCH}")



In [ ]:
# run base policy rollouts
GLOBAL_SEED = 2
ROBOT_STATE_OVERRIDE = None
NUM_ROLLOUTS = 10
HORIZON = 100
STOP_ON_BEHAVIOR = True
FOR_DISPLAY_STOP = False

if "policy" not in globals() or "env" not in globals():
    raise RuntimeError("Run the policy loading cell before running rollouts.")
if not is_env_connected(env):
    if "ckpt_dict" not in globals():
        raise RuntimeError("Loaded env is disconnected and ckpt_dict is unavailable. Rerun the policy loading cell.")
    print("Loaded env is disconnected; recreating env from checkpoint metadata.")
    env, _ = FileUtils.env_from_checkpoint(
        ckpt_dict=ckpt_dict,
        render=False,
        render_offscreen=True,
        verbose=True,
    )
    base_env_state = {key: np.asarray(value, dtype=np.float32).copy() for key, value in env.get_state().items()}

video_cfg = load_json_config(VISUALIZATION_CONFIG_PATH)
fixed_scene, fixed_robot, scene_cfg = fixed_scene_robot_from_config(base_env_state, SCENE_CONFIG_PATH, ROBOT_STATE_OVERRIDE)

output_root = Path(OUTPUT_ROOT)
if not output_root.is_absolute():
    output_root = REPO_ROOT / output_root

scene_name = scene_cfg["name"]
fps = int(VIDEO_FPS)
horizon = int(HORIZON)
num_rollouts = int(NUM_ROLLOUTS)
stop_on_behavior = bool(STOP_ON_BEHAVIOR)
for_display_stop = bool(FOR_DISPLAY_STOP)

run_id = run_folder_name(
    "base_policy",
    f"policy_{POLICY_EPOCH}",
    f"scene_{scene_name}",
    f"rollouts{num_rollouts}",
    f"horizon{horizon}",
    f"seed{GLOBAL_SEED if GLOBAL_SEED is not None else 'none'}",
)
policy_rollout_out_dir = output_root / run_id
policy_rollout_out_dir.mkdir(parents=True, exist_ok=True)
policy_rollout_summaries = []

seed_everything(GLOBAL_SEED)
for rollout_idx in range(num_rollouts):
    rollout_seed = None if GLOBAL_SEED is None else GLOBAL_SEED + rollout_idx
    seed_everything(rollout_seed)
    policy.start_episode()
    obs = reset_env_to_scene_robot(env, fixed_scene, fixed_robot)
    scene_snapshot = capture_scene_snapshot(env)
    start_state = env.get_state()
    start_scene = np.asarray(start_state["scene"], dtype=np.float32).copy()
    binaries = articulated_binaries_from_start_state(start_scene)

    frames = [render_visual_camera(env, video_cfg)]
    actions, rewards, dones = [], [], []
    scene_states = [start_scene.copy()]
    robot_states = [np.asarray(start_state["robot"], dtype=np.float32).copy()]
    eef_xy = [robot_states[-1][:2].copy()]

    detected_behavior = "none"
    detected_step = -1
    termination_reason = "horizon"

    for step in range(horizon):
        action = policy(ob=obs)
        obs, reward, done, info = env.step(action)
        state = env.get_state()
        scene = np.asarray(state["scene"], dtype=np.float32).copy()
        robot = np.asarray(state["robot"], dtype=np.float32).copy()

        actions.append(np.asarray(action, dtype=np.float32).copy())
        rewards.append(float(reward))
        dones.append(bool(done))
        scene_states.append(scene)
        robot_states.append(robot)
        eef_xy.append(robot[:2].copy())
        frames.append(render_visual_camera(env, video_cfg))

        triggered = check_state_difference(start_scene, scene, robot[:3], binaries, for_display=for_display_stop)
        if detected_step < 0 and triggered:
            detected_behavior = classify_behavior(start_scene, scene, robot[:3], binaries, for_display=for_display_stop)
            detected_step = step + 1
            if stop_on_behavior:
                termination_reason = "behavior"
                break
        if done:
            termination_reason = "env_done"
            break

    rollout = {
        "scene_config": scene_name,
        "seed": rollout_seed,
        "behavior": detected_behavior,
        "behavior_step": detected_step,
        "termination_step": len(actions),
        "termination_reason": termination_reason,
        "return": float(np.sum(rewards)),
        "actions": np.asarray(actions, dtype=np.float32),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "dones": np.asarray(dones, dtype=bool),
        "scene_states": np.asarray(scene_states, dtype=np.float32),
        "robot_states": np.asarray(robot_states, dtype=np.float32),
        "eef_xy": np.asarray(eef_xy, dtype=np.float32),
        "scene_snapshot": scene_snapshot,
    }
    save_rollout_artifacts(rollout, frames, policy_rollout_out_dir, f"rollout_{rollout_idx:03d}", video_cfg, fps=fps)

    policy_rollout_summaries.append({
        "scene_config": scene_name,
        "rollout": rollout_idx,
        "seed": rollout_seed,
        "behavior": detected_behavior,
        "step": detected_step,
        "termination_step": len(actions),
        "termination_reason": termination_reason,
        "video": rollout["video"],
        "trace": rollout["trace"],
        "scene_snapshot": rollout["scene_snapshot_path"],
    })

summary_path = policy_rollout_out_dir / "policy_rollout_summary.json"
with open(summary_path, "w") as f:
    json.dump([{key: str(value) if isinstance(value, Path) else value for key, value in row.items()} for row in policy_rollout_summaries], f, indent=2)

print(f"Policy rollout output: {policy_rollout_out_dir}")
print(f"Summary JSON: {summary_path}")
print("Detected behaviors:")
for row in policy_rollout_summaries:
    print(
        f"{row['scene_config']} rollout {row['rollout']:03d} seed {row['seed']} -> "
        f"{row['behavior']} @ detected step {row['step']} | "
        f"terminated step {row['termination_step']} ({row['termination_reason']}) | "
        f"video: {row['video']}"
    )

if policy_rollout_summaries:
    try:
        from IPython.display import Video, display
        display(Video(filename=str(policy_rollout_summaries[0]["video"]), embed=True))
    except Exception as exc:
        print(f"Inline video display skipped: {exc}")



In [ ]:
# plot rollout xy overlay
if "policy_rollout_summaries" not in globals() or not policy_rollout_summaries:
    print("Run the policy rollout cell above first; it writes trace .npz files for plotting.")
else:
    plot_rollouts_from_trace_summaries(policy_rollout_summaries)
